# Analytics for analyzing the survey data collected via the survey js questionnaires

In [1]:
#%pip install pymongo python-dotenv pandas

In [2]:
# importing packages
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from pymongo import MongoClient
import os


## Getting the data from mongo db 

You need to connect to the mongoDB first. Mongo is only listening inside the docker network. So you need to connect via ssh to the dockerNetwork where the DB is running.

1. Go to ~/.ssh/config
2. Open the file
3. add this entry 

` # Habit Hub Mongo DB
Host habitHubMongo
  HostName 141.76.16.16
  User service
  LocalForward 27017 172.18.0.4:27017 `

4. save the file
5. run `ssh habitHubMongo` in your terminal
6. keep this terminal session open


In [3]:
# load repo .env
repo_root = Path("..")  # notebook is in analytics/
load_dotenv(repo_root / ".env")

host = os.getenv("MONGO_HOST", "localhost")
port = int(os.getenv("MONGO_PORT", "27017"))
user = os.getenv("MONGO_USER")
password = os.getenv("MONGO_PASSWORD")
db_name = os.getenv("MONGO_DB", "surveyjs")
auth_source = os.getenv("MONGO_AUTH_SOURCE", "admin")

uri = f"mongodb://{user}:{password}@{host}:{port}/?authSource={auth_source}"

client = MongoClient(
  uri,
  serverSelectionTimeoutMS=int(os.getenv("MONGO_SERVER_SELECTION_TIMEOUT_MS", "5000")),
  socketTimeoutMS=int(os.getenv("MONGO_SOCKET_TIMEOUT_MS", "5000")),
)

db = client[db_name]
collection = db["results"]

docs = list(collection.find({}))  # all documents right now
len(docs)


print(client.list_database_names())
print(db_name, db.list_collection_names())
print(collection.count_documents({}))


['admin', 'config', 'local', 'surveyjs']
surveyjs ['surveys', 'results']
33


## Prepare dataset (prod only) and flatten responses
This section normalizes the nested response payloads, keeps only production submissions, and
treats `-1` as missing.


In [ ]:
# Flatten prod responses from docs
rows = []
for d in docs:
    payload = (d.get('data') or {}).get('data') or d.get('json') or {}
    source = 'prod' if (d.get('data') or {}).get('data') else 'test'
    rows.append({
        **payload,
        '_id': str(d.get('_id')),
        'surveyId': d.get('surveyId') or (d.get('data') or {}).get('surveyId'),
        'submittedAt': d.get('submittedAt') or (d.get('data') or {}).get('completedAt'),
        'userId': d.get('userId'),
        'source': source,
    })

df = pd.DataFrame(rows)
df = df[df['source'] == 'prod'].copy()

# Treat -1 as missing for all numeric questionnaire items
sus_cols = [f'sus_q{i}' for i in range(1, 11)]
ueq_cols = [
    'annoying_enjoyable',
    'not_understandable_understandable',
    'creative_dull',
    'easy_to_learn_difficult_to_learn',
    'valuable_inferior',
    'boring_exciting',
    'not_interesting_interesting',
    'unpredictable_predictable',
    'fast_slow',
    'inventive_conventional',
    'obstructive_supportive',
    'good_bad',
    'complicated_easy',
    'unlikable_pleasing',
    'usual_leading_edge',
    'unpleasant_pleasant',
    'secure_not_secure',
    'motivating_demotivating',
    'meets_expectations_does_not_meet_expectations',
    'inefficient_efficient',
    'clear_confusing',
    'impractical_practical',
    'organized_cluttered',
    'attractive_unattractive',
    'friendly_unfriendly',
    'conservative_innovative',
]

all_items = sus_cols + ueq_cols
df[all_items] = df[all_items].replace(-1, np.nan).astype(float) #replace -1 with NaN
df.shape


(30, 53)

## SUS scoring (0-100)
Standard SUS: odd items are positive, even items are reversed. Scores are scaled to 0-100.


In [ ]:
sus_pos = ['sus_q1', 'sus_q3', 'sus_q5', 'sus_q7', 'sus_q9']
sus_neg = ['sus_q2', 'sus_q4', 'sus_q6', 'sus_q8', 'sus_q10']

sus = df[sus_cols].copy()
sus_score = pd.DataFrame(index=df.index)
sus_score[sus_pos] = sus[sus_pos] - 1
sus_score[sus_neg] = 5 - sus[sus_neg]

sus_total = sus_score.sum(axis=1, skipna=True)
sus_complete = sus_score.notna().sum(axis=1) == 10
sus_total[~sus_complete] = np.nan

df['sus_score'] = sus_total * 2.5

df['sus_score'].describe()


## UEQ scoring (6 scales)
UEQ items are mapped from 1-7 to -3..+3. Items where the positive pole is on the left are reversed.


In [ ]:
# Items where the positive pole is on the right (negative -> positive)
pos_right = {
    'annoying_enjoyable',
    'not_understandable_understandable',
    'boring_exciting',
    'not_interesting_interesting',
    'unpredictable_predictable',
    'obstructive_supportive',
    'complicated_easy',
    'unlikable_pleasing',
    'usual_leading_edge',
    'unpleasant_pleasant',
    'inefficient_efficient',
    'impractical_practical',
    'conservative_innovative',
}

pos_left = set(ueq_cols) - pos_right

ueq_raw = df[ueq_cols].copy()
ueq_scored = pd.DataFrame(index=df.index)

# Positive on right: 1..7 -> -3..3
ueq_scored[list(pos_right)] = ueq_raw[list(pos_right)] - 4
# Positive on left: reverse (1..7 -> 3..-3)
ueq_scored[list(pos_left)] = 4 - ueq_raw[list(pos_left)]

ueq_scored.head()


In [ ]:
ueq_scales = {
    'Attractiveness': [
        'annoying_enjoyable',
        'good_bad',
        'unlikable_pleasing',
        'unpleasant_pleasant',
        'attractive_unattractive',
        'friendly_unfriendly',
    ],
    'Perspicuity': [
        'not_understandable_understandable',
        'easy_to_learn_difficult_to_learn',
        'complicated_easy',
        'clear_confusing',
    ],
    'Efficiency': [
        'inefficient_efficient',
        'fast_slow',
        'impractical_practical',
        'organized_cluttered',
    ],
    'Dependability': [
        'unpredictable_predictable',
        'obstructive_supportive',
        'secure_not_secure',
        'meets_expectations_does_not_meet_expectations',
    ],
    'Stimulation': [
        'boring_exciting',
        'not_interesting_interesting',
        'motivating_demotivating',
        'valuable_inferior',
    ],
    'Novelty': [
        'creative_dull',
        'inventive_conventional',
        'usual_leading_edge',
        'conservative_innovative',
    ],
}

ueq_scale_scores = pd.DataFrame(index=df.index)
for scale, items in ueq_scales.items():
    vals = ueq_scored[items]
    min_items = int(np.ceil(len(items) / 2))
    scores = vals.mean(axis=1, skipna=True)
    scores[vals.notna().sum(axis=1) < min_items] = np.nan
    ueq_scale_scores[scale] = scores

df = df.join(ueq_scale_scores)
df[['Attractiveness','Perspicuity','Efficiency','Dependability','Stimulation','Novelty']].describe()


## Visuals
We show SUS distribution, UEQ scale means with 95% CI, and item-level means.


In [ ]:
# SUS distribution
plt.figure(figsize=(8, 4))
sns.histplot(df['sus_score'].dropna(), bins=10, kde=True)
plt.title('SUS Score Distribution (0-100)')
plt.xlabel('SUS Score')
plt.ylabel('Count')
plt.show()

# UEQ scale means with 95% CI
scale_stats = []
for scale in ueq_scales.keys():
    s = df[scale].dropna()
    n = len(s)
    mean = s.mean()
    sd = s.std(ddof=1)
    ci = 1.96 * sd / np.sqrt(n) if n > 1 else np.nan
    scale_stats.append((scale, mean, ci, n))

scale_df = pd.DataFrame(scale_stats, columns=['scale', 'mean', 'ci95', 'n'])

plt.figure(figsize=(9, 5))
plt.bar(scale_df['scale'], scale_df['mean'], yerr=scale_df['ci95'], capsize=5)
plt.axhline(0, color='gray', linewidth=1)
plt.ylim(-3, 3)
plt.title('UEQ Scale Means (-3 to +3) with 95% CI')
plt.ylabel('Mean score')
plt.xticks(rotation=30, ha='right')
plt.show()

# Item-level means (UEQ)
item_means = ueq_scored.mean().sort_values()
plt.figure(figsize=(9, 8))
item_means.plot(kind='barh')
plt.axvline(0, color='gray', linewidth=1)
plt.title('UEQ Item Means (-3 to +3)')
plt.xlabel('Mean score')
plt.tight_layout()
plt.show()


## Interpretation (automatic summary)
Below we summarize the SUS and UEQ results, including a benchmark classification for web sites/services.


In [ ]:
# UEQ benchmark for web sites and web services (from handbook)
benchmark = {
    'Attractiveness': [1.75, 1.41, 0.96, 0.44],
    'Perspicuity': [2.07, 1.84, 1.14, 0.65],
    'Efficiency': [1.70, 1.43, 0.98, 0.50],
    'Dependability': [1.70, 1.53, 1.19, 0.81],
    'Stimulation': [1.56, 1.10, 0.69, 0.07],
    'Novelty': [1.12, 0.87, 0.49, -0.22],
}

def benchmark_label(scale, mean):
    if pd.isna(mean):
        return 'n/a'
    excellent, good, above_avg, below_avg = benchmark[scale]
    if mean >= excellent:
        return 'Excellent'
    if mean >= good:
        return 'Good'
    if mean >= above_avg:
        return 'Above average'
    if mean >= below_avg:
        return 'Below average'
    return 'Bad'

summary = scale_df.copy()
summary['benchmark'] = summary.apply(lambda r: benchmark_label(r['scale'], r['mean']), axis=1)
summary


In [ ]:
# Text summary for quick reporting
sus_mean = df['sus_score'].mean()
sus_n = df['sus_score'].notna().sum()

print('SUS mean: {:.1f} (n={})'.format(sus_mean, sus_n))
print('Rule of thumb: SUS > 68 is above average.')

for _, r in summary.iterrows():
    print('{}: {:.2f} (n={}), {}'.format(r['scale'], r['mean'], int(r['n']), r['benchmark']))


In [ ]:
from IPython.display import Markdown, display

n_total = len(df)
sus_mean = df['sus_score'].mean()
sus_n = df['sus_score'].notna().sum()
sus_label = 'above average' if sus_mean > 68 else 'below average'

missing_rate = 1 - ueq_scored.notna().mean().mean()

lines = []
lines.append('### Interpretation')
lines.append('')
lines.append('- Sample size: {} production responses.'.format(n_total))
lines.append('- Missing UEQ item rate: {:.1%} (skipped answers are treated as missing).'.format(missing_rate))
lines.append('- SUS mean: {:.1f} (n={}), which is {} vs the common 68-point benchmark.'.format(sus_mean, sus_n, sus_label))
lines.append('')
lines.append('UEQ scale interpretation (web sites/services benchmark):')
for _, r in summary.iterrows():
    lines.append('- {}: {:.2f} (n={}), {}'.format(r['scale'], r['mean'], int(r['n']), r['benchmark']))

lines.append('')
lines.append('Notes:')
lines.append('- Scale means are on the -3..+3 UEQ scale; values between -0.8 and 0.8 are neutral.')
lines.append('- Larger confidence intervals indicate higher uncertainty (often from small sample sizes).')

display(Markdown('\n'.join(lines)))
